# LocalTransform 环境配置教程

## 概述

LocalTransform 是一种基于**局部反应模板 + 图神经网络**的正向反应产物预测方法，发表于 Nature Machine Intelligence 2022。

> **论文**: *A Generalized Template-Based Graph Neural Network for Accurate Organic Reactivity Prediction* (Chen & Jung, 2022)  
> **核心思想**: 不同于全局反应模板方法（如 GLN），LocalTransform 将反应预测分解为**键级别的二分类（是否反应性）+ 模板分类（应用哪个局部模板）**。每条化学键（包括虚拟键和真实键）都被独立评估，然后通过全局注意力机制聚合局部信息。

### 本教程包含三份 notebook

| 序号 | Notebook | 内容 |
|------|----------|------|
| 1 | **环境配置**（本文件） | 安装依赖、验证环境、源码结构概览 |
| 2 | 数据处理 | 最小化的 LocalTransform 数据处理流程 |
| 3 | 模型展示 | LocalTransform 推理原理与模型计算架构 |

---

## 1. 核心依赖一览

LocalTransform 基于 DGL（Deep Graph Library）而非 PyG 构建图神经网络。以下是依赖列表及教学版本适配：

| 依赖库 | 原始版本 | 教程版本（推荐） | 说明 |
|--------|----------|-----------------|------|
| Python | 3.6+ | **3.10+** | 现代化 Python |
| PyTorch | 1.0+ | **2.0+（CPU）** | 深度学习框架（仅CPU足够教学） |
| DGL | 0.5.2+ | **2.0+** | 图神经网络框架 |
| DGL-Life | 0.2.6+ | **0.3.2+** | DGL 生命科学扩展（分子特征化） |
| RDKit | 2019+ | **2023.09+** | 化学信息学工具包 |
| NumPy | 1.16+ | **1.24+** | 数值计算 |
| Pandas | — | **1.5+** | 数据处理 |

> **与 GLN 的区别**: GLN 使用 PyTorch Geometric (torch-geometric) 作为 GNN 框架，而 LocalTransform 使用 DGL + DGL-Life。DGL-Life 提供了开箱即用的分子特征化工具（WeaveAtomFeaturizer、CanonicalBondFeaturizer）和 MPNN 模型。

> **关于 GPU**: 本教程仅用于教学演示，使用 CPU 版本的 PyTorch 即可。不做训练。

## 2. 创建虚拟环境并安装依赖

以下代码会在项目的 `envs/` 目录下创建一个独立的 conda 虚拟环境，避免与系统环境冲突。

In [1]:
import os
from pathlib import Path

# ====== 定位项目根目录 ======
def find_project_root(start=None):
    """向上查找项目根目录（包含 teaching_demos/ 的目录）"""
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
ENV_DIR = PROJECT_ROOT / 'envs' / 'localtransform_tutorial_envs'
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalTransform'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'虚拟环境目录: {ENV_DIR}')
print(f'LocalTransform 源码目录: {SOURCE_REPO}')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
虚拟环境目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs
LocalTransform 源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform


In [2]:
%%bash -s "$ENV_DIR"

ENV_DIR=$1

if ! command -v conda >/dev/null 2>&1; then
    for candidate in "$HOME/miniconda3/etc/profile.d/conda.sh" "$HOME/anaconda3/etc/profile.d/conda.sh" "/opt/conda/etc/profile.d/conda.sh"; do
        if [ -f "$candidate" ]; then
            # shellcheck disable=SC1090
            source "$candidate"
            break
        fi
    done
fi

if ! command -v conda >/dev/null 2>&1; then
    echo "未找到 conda，请先安装 Miniconda/Anaconda。" >&2
    exit 1
fi

if [ ! -d "$ENV_DIR" ]; then
    echo "====== 创建 conda 环境 ======"
    conda create -y -p "$ENV_DIR" python=3.10
else
    echo "====== 环境已存在，执行增量修复 ======"
fi

echo "====== 安装 PyTorch (CPU) ======"
conda install -y -p "$ENV_DIR" pytorch cpuonly -c pytorch

echo "====== 安装 DGL ======"
conda install -y -p "$ENV_DIR" -c dglteam/label/th24_cu121 dgl

echo "====== 安装 RDKit / DGL-Life / 常用依赖 ======"
conda install -y -p "$ENV_DIR" -c conda-forge dgllife rdkit numpy pandas scikit-learn tqdm ipykernel

echo "====== 修复 PyTorch + MKL 兼容性 ======"
conda install -y -p "$ENV_DIR" "mkl<2025" "intel-openmp<2025"

echo "====== 安装协作式 JupyterLab 组件（用于 MCP / Notebook 联调） ======"
"$ENV_DIR/bin/python" -m pip install \
    'jupyterlab==4.4.7' \
    'jupyter-collaboration==4.1.1' \
    'jupyter-server-fileid==0.9.3' \
    'jupyter-server-terminals==0.5.3' \
    'socksio>=1.0.0'

echo "====== 注册 Jupyter Kernel ======"
"$ENV_DIR/bin/python" -m ipykernel install --user \
    --name localtransform_tutorial_envs \
    --display-name "Python (localtransform_tutorial_envs)"

echo "====== 验证核心导入 ======"
"$ENV_DIR/bin/python" - <<'PY'
import torch, dgl, dgllife, rdkit, numpy, pandas
print('torch', torch.__version__)
print('dgl', dgl.__version__)
print('dgllife', dgllife.__version__)
print('rdkit', rdkit.__version__)
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
PY

echo "====== 环境创建/修复完成 ======"

====== 环境已存在，执行增量修复 ======
====== 安装 PyTorch (CPU) ======


Channels:
 - pytorch
 - defaults
 - conda-forge
 - dglteam/label/th24_cu121
Platform: linux-64
Colle

cting package metadata (repodata.json): ...working... 

done
Solving environment: ...working... 

done


...

Installed kernelspec localtransform_tutorial_envs in /home/xiaoruiwang/.local/share/jupyter/kernels/

localtransform_tutorial_envs


====== 验证核心导入 ======


torch 2.5.1
dgl 2.4.0+cu121
dgllife 0.3.2
rdkit 2025.09.6
numpy 2.2.5
pandas 2.3.3


====== 环境创建/修复完成 ======


## 3. 验证环境

以下代码验证核心库是否正确安装，并打印版本信息。

> **注意**: 如果在 Jupyter 中执行，请确保 kernel 已切换到刚创建的 `localtransform_tutorial_envs` 环境。

In [3]:
# ====== Step 3.1: 验证核心导入 ======
import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__} (CUDA available: {torch.cuda.is_available()})')

import dgl
print(f'DGL: {dgl.__version__}')

import dgllife
print(f'DGL-Life: {dgllife.__version__}')

import rdkit
print(f'RDKit: {rdkit.__version__}')

import numpy as np
print(f'NumPy: {np.__version__}')

import pandas as pd
print(f'Pandas: {pd.__version__}')

Python: 3.10.12 | packaged by conda-forge | (main, Jun 23 2023, 22:40:32) [GCC 12.3.0]


PyTorch: 2.5.1 (CUDA available: False)


/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DGL: 2.4.0+cu121


DGL-Life: 0.3.2
RDKit: 2025.09.6
NumPy: 2.2.5
Pandas: 2.3.3


In [4]:
# ====== Step 3.2: 验证 DGL-Life 分子特征化器 ======
# LocalTransform 的核心特征化工具来自 dgllife
from dgllife.utils import WeaveAtomFeaturizer, CanonicalBondFeaturizer, mol_to_bigraph
from rdkit import Chem

# 用一个简单分子测试
mol = Chem.MolFromSmiles('CCO')  # 乙醇
atom_featurizer = WeaveAtomFeaturizer()
bond_featurizer = CanonicalBondFeaturizer(self_loop=True)

g = mol_to_bigraph(mol, 
                   node_featurizer=atom_featurizer, 
                   edge_featurizer=bond_featurizer,
                   add_self_loop=True,
                   canonical_atom_order=False)

print(f'DGL 图: {g}')
print(f'节点特征维度: {g.ndata["h"].shape}')  # (num_atoms, feat_dim)
print(f'边特征维度: {g.edata["e"].shape}')    # (num_edges, feat_dim)
print('\n✅ DGL-Life 分子特征化正常工作！')

DGL 图: Graph(num_nodes=3, num_edges=7,
      ndata_schemes={'h': Scheme(shape=(27,), dtype=torch.float32)}
      edata_schemes={'e': Scheme(shape=(13,), dtype=torch.float32)})
节点特征维度: torch.Size([3, 27])
边特征维度: torch.Size([7, 13])

✅ DGL-Life 分子特征化正常工作！


## 4. LocalTransform 源码结构

### 为什么需要了解源码结构？

后续的数据处理和模型展示 notebook 会频繁引用源码中的文件和函数。先了解整体架构有助于建立全局认知。

In [5]:
# ====== Step 4.1: 展示源码目录结构 ======
import os
from pathlib import Path

def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalTransform'

def show_tree(root, prefix='', max_depth=3, _depth=0):
    """递归打印目录树（跳过隐藏目录和缓存）"""
    if _depth >= max_depth:
        return
    entries = sorted(root.iterdir())
    dirs = [e for e in entries if e.is_dir() and not e.name.startswith('.') and e.name != '__pycache__']
    files = [e for e in entries if e.is_file() and not e.name.startswith('.')]
    for f in files:
        size = f.stat().st_size
        if size > 1024*1024:
            size_str = f'{size/1024/1024:.1f} MB'
        elif size > 1024:
            size_str = f'{size/1024:.1f} KB'
        else:
            size_str = f'{size} B'
        print(f'{prefix}├── {f.name}  ({size_str})')
    for d in dirs:
        print(f'{prefix}├── {d.name}/')
        show_tree(d, prefix + '│   ', max_depth, _depth + 1)

print(f'LocalTransform 源码目录: {SOURCE_REPO}')
print('=' * 60)
show_tree(SOURCE_REPO)

LocalTransform 源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/LocalTransform
├── README.md  (4.5 KB)
├── Synthesis.ipynb  (2.5 MB)
├── Synthesis.py  (9.3 KB)
├── LocalTemplate/
│   ├── __init__.py  (0 B)
│   ├── template_collector.py  (11.2 KB)
│   ├── template_decoder.py  (11.0 KB)
│   ├── template_extract_utils.py  (13.4 KB)
│   ├── template_extractor.py  (26.4 KB)
├── data/
│   ├── USPTO_480k/
│   │   ├── preprocessed_test.csv  (18.9 MB)
│   │   ├── real_templates.csv  (357.7 KB)
│   │   ├── template_infos.csv  (614.9 KB)
│   │   ├── virtual_templates.csv  (206.3 KB)
│   ├── configs/
│   │   ├── default_config.json  (136 B)
├── models/
│   ├── LocalTransform_mix.pth  (34.7 MB)
├── preprocessing/
│   ├── Extract_from_train_data.py  (5.7 KB)
│   ├── Run_preprocessing.py  (8.7 KB)
│   ├── __init__.py  (0 B)
├── scripts/
│   ├── Calculate_topk_accuracy.py  (3.0 KB)
│   ├── Decode_predictions.py  (3.7 KB)
│   ├── Test.py  (2.1 KB)
│   

In [6]:
# ====== Step 4.2: 源码模块职责表 ======
module_info = [
    ('Synthesis.py',              '推理入口', '提供 localtransform 类，封装完整的预测流程'),
    ('scripts/models.py',         '模型定义', 'LocalTransform 主模型类（MPNN + 注意力 + 分类头）'),
    ('scripts/model_utils.py',    '模型工具', 'MultiHeadAttention、反应性池化、特征打包/解包'),
    ('scripts/dataset.py',        '数据集', 'USPTODataset/USPTOTestDataset（SMILES → DGL图）'),
    ('scripts/utils.py',          '通用工具', '特征化器初始化、模型加载、数据整理、预测包装'),
    ('scripts/get_edit.py',       '编辑提取', '将模型输出转换为键级别的预测结果'),
    ('scripts/Train.py',          '训练脚本', '训练循环、损失计算、学习率调度'),
    ('scripts/Test.py',           '测试脚本', '在测试集上生成预测'),
    ('scripts/Decode_predictions.py', '解码', '将原始预测解码为产物 SMILES'),
    ('preprocessing/',            '预处理', '模板提取 + 数据标注'),
    ('LocalTemplate/',            '模板模块', '模板提取、解码、应用（生成产物）'),
    ('data/configs/',             '配置文件', '模型超参数配置 JSON'),
    ('data/USPTO_480k/',          '数据目录', '模板文件、预处理数据'),
    ('models/',                   '模型权重', '预训练模型 .pth 文件'),
]

print(f'{"文件/目录":<35} {"角色":<12} {"说明"}')
print('=' * 90)
for path, role, desc in module_info:
    print(f'{path:<35} {role:<12} {desc}')

文件/目录                               角色           说明
Synthesis.py                        推理入口         提供 localtransform 类，封装完整的预测流程
scripts/models.py                   模型定义         LocalTransform 主模型类（MPNN + 注意力 + 分类头）
scripts/model_utils.py              模型工具         MultiHeadAttention、反应性池化、特征打包/解包
scripts/dataset.py                  数据集          USPTODataset/USPTOTestDataset（SMILES → DGL图）
scripts/utils.py                    通用工具         特征化器初始化、模型加载、数据整理、预测包装
scripts/get_edit.py                 编辑提取         将模型输出转换为键级别的预测结果
scripts/Train.py                    训练脚本         训练循环、损失计算、学习率调度
scripts/Test.py                     测试脚本         在测试集上生成预测
scripts/Decode_predictions.py       解码           将原始预测解码为产物 SMILES
preprocessing/                      预处理          模板提取 + 数据标注
LocalTemplate/                      模板模块         模板提取、解码、应用（生成产物）
data/configs/                       配置文件         模型超参数配置 JSON
data/USPTO_480k/                    数据目录         模板文件、预处理数据
models/                     

## 5. LocalTransform 整体工作流

### 核心思想

LocalTransform 的关键创新在于将反应预测问题转化为**键级别的局部模板分类**问题：

```
输入: 反应物 SMILES (如 "CCO.HCl")
  │
  ├── [1] 分子图构建
  │     ├── 特征图 (Feature Graph): DGL 图 + 原子/键特征
  │     └── 稠密图 (Dense Graph): 原子距离矩阵 + 虚拟键/真实键列表
  │
  ├── [2] GNN 编码 (MPNN 消息传递)
  │     └── 原子特征 → 消息传递 → 更新后的原子表示
  │
  ├── [3] 全局反应性注意力 (Global Reactivity Attention)
  │     ├── 原子级注意力: 利用距离矩阵作为位置编码
  │     └── 键级注意力: 利用键距离矩阵
  │
  ├── [4] 键级别预测
  │     ├── 反应性预测: 每条键是否参与反应? (二分类)
  │     ├── 反应性池化: 选出 Top-K 最可能反应的键
  │     └── 模板分类: 对 Top-K 键分类应该用哪个局部模板
  │
  ├── [5] 后处理
  │     ├── 合并虚拟键和真实键的预测
  │     ├── 按分数排序取 Top-K
  │     └── 应用局部模板生成产物 SMILES
  │
  └── 输出: Top-K 预测产物
```

### 与其他方法的对比

| 维度 | GLN (逆合成) | LocalTransform (正向) |
|------|-------------|---------------------|
| 预测方向 | 产物 → 反应物 | 反应物 → 产物 |
| 模板粒度 | 全局反应模板 | 局部键级别模板 |
| GNN 框架 | PyTorch Geometric | DGL + DGL-Life |
| 预测目标 | 整个分子的模板 | 每条键的模板 |
| 键类型 | — | 虚拟键(virtual) + 真实键(real) |

In [7]:
# ====== Step 5.1: 理解"虚拟键"和"真实键"的概念 ======
# 这是 LocalTransform 的核心概念之一

from rdkit import Chem
import numpy as np

smiles = 'CCO'  # 乙醇: C-C-O
mol = Chem.MolFromSmiles(smiles)

# 真实键 (Real bonds): 分子中已存在的化学键
real_bonds = []
for atom in mol.GetAtoms():
    for bond in atom.GetBonds():
        pair = (bond.GetBeginAtom().GetIdx(), bond.GetEndAtom().GetIdx())
        real_bonds.append(pair)

# 虚拟键 (Virtual bonds): 所有未成键的原子对
n_atoms = mol.GetNumAtoms()
all_pairs = [(i, j) for i in range(n_atoms) for j in range(n_atoms) if i != j]
real_set = set(real_bonds)
virtual_bonds = [(i, j) for (i, j) in all_pairs if (i, j) not in real_set]

print(f'分子: {smiles}')
print(f'原子数: {n_atoms}')
print(f'\n真实键 (Real bonds) — 已有的化学键:')
for (a, b) in real_bonds:
    atom_a = mol.GetAtomWithIdx(a).GetSymbol()
    atom_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  {atom_a}({a}) — {atom_b}({b})')

print(f'\n虚拟键 (Virtual bonds) — 未成键的原子对:')
for (a, b) in virtual_bonds:
    atom_a = mol.GetAtomWithIdx(a).GetSymbol()
    atom_b = mol.GetAtomWithIdx(b).GetSymbol()
    print(f'  {atom_a}({a}) ··· {atom_b}({b})')

print(f'\n> 真实键数: {len(real_bonds)}, 虚拟键数: {len(virtual_bonds)}')
print('> 真实键对应"键断裂/变化"反应, 虚拟键对应"键形成"反应')

分子: CCO
原子数: 3

真实键 (Real bonds) — 已有的化学键:
  C(0) — C(1)
  C(0) — C(1)
  C(1) — O(2)
  C(1) — O(2)

虚拟键 (Virtual bonds) — 未成键的原子对:
  C(0) ··· O(2)
  C(1) ··· C(0)
  O(2) ··· C(0)
  O(2) ··· C(1)

> 真实键数: 4, 虚拟键数: 4
> 真实键对应"键断裂/变化"反应, 虚拟键对应"键形成"反应


In [8]:
# ====== Step 5.2: 理解原子距离矩阵 (ADM) ======
# 原子距离矩阵用于为注意力机制提供位置编码

from rdkit import Chem
import numpy as np

smiles = 'c1ccccc1O'  # 苯酚
mol = Chem.MolFromSmiles(smiles)

# 获取拓扑距离矩阵
dm = Chem.GetDistanceMatrix(mol)
print(f'分子: {smiles}')
print(f'原子列表: {[atom.GetSymbol() for atom in mol.GetAtoms()]}')
print(f'\n原始距离矩阵 (拓扑距离，即最短路径上的键数):')
print(dm.astype(int))

# LocalTransform 的处理: 截断到 max_distance=6，超远设为 7
max_distance = 6
adm = dm.copy()
adm[adm > 100] = -1         # 不同分子 → 远程
adm[adm > max_distance] = max_distance  # 同一分子内的远程
adm[adm == -1] = max_distance + 1       # 不同分子 → max+1

print(f'\n处理后的原子距离矩阵 (截断到 {max_distance}，不同分子为 {max_distance+1}):')
print(adm.astype(int))
print(f'\n> 距离值 0-{max_distance}: 拓扑距离')
print(f'> 距离值 {max_distance+1}: 位于不同分子片段（如多组分反应物）')

分子: c1ccccc1O
原子列表: ['C', 'C', 'C', 'C', 'C', 'C', 'O']

原始距离矩阵 (拓扑距离，即最短路径上的键数):
[[0 1 2 3 2 1 2]
 [1 0 1 2 3 2 3]
 [2 1 0 1 2 3 4]
 [3 2 1 0 1 2 3]
 [2 3 2 1 0 1 2]
 [1 2 3 2 1 0 1]
 [2 3 4 3 2 1 0]]

处理后的原子距离矩阵 (截断到 6，不同分子为 7):
[[0 1 2 3 2 1 2]
 [1 0 1 2 3 2 3]
 [2 1 0 1 2 3 4]
 [3 2 1 0 1 2 3]
 [2 3 2 1 0 1 2]
 [1 2 3 2 1 0 1]
 [2 3 4 3 2 1 0]]

> 距离值 0-6: 拓扑距离
> 距离值 7: 位于不同分子片段（如多组分反应物）


## 6. 验证预训练模型文件

LocalTransform 提供了在 USPTO_480k 数据集上预训练的模型权重，我们后续在模型展示 notebook 中会加载它做推理演示。

In [9]:
# ====== Step 6.1: 检查数据文件和模型权重 ======
import os
from pathlib import Path

def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalTransform'

expected_files = {
    '模型权重': SOURCE_REPO / 'models' / 'LocalTransform_mix.pth',
    '真实键模板': SOURCE_REPO / 'data' / 'USPTO_480k' / 'real_templates.csv',
    '虚拟键模板': SOURCE_REPO / 'data' / 'USPTO_480k' / 'virtual_templates.csv',
    '模板信息': SOURCE_REPO / 'data' / 'USPTO_480k' / 'template_infos.csv',
    '模型配置': SOURCE_REPO / 'data' / 'configs' / 'default_config.json',
    '测试数据': SOURCE_REPO / 'data' / 'USPTO_480k' / 'preprocessed_test.csv',
}

print(f'{"文件":<15} {"状态":<6} {"大小":<12} 路径')
print('=' * 90)
for name, path in expected_files.items():
    if path.exists():
        size = path.stat().st_size
        if size > 1024*1024:
            size_str = f'{size/1024/1024:.1f} MB'
        elif size > 1024:
            size_str = f'{size/1024:.1f} KB'
        else:
            size_str = f'{size} B'
        print(f'{name:<15} ✅     {size_str:<12} {path.relative_to(PROJECT_ROOT)}')
    else:
        print(f'{name:<15} ❌     —            {path.relative_to(PROJECT_ROOT)}')

文件              状态     大小           路径
模型权重            ✅     34.7 MB      source_repos/LocalTransform/models/LocalTransform_mix.pth
真实键模板           ✅     357.7 KB     source_repos/LocalTransform/data/USPTO_480k/real_templates.csv
虚拟键模板           ✅     206.3 KB     source_repos/LocalTransform/data/USPTO_480k/virtual_templates.csv
模板信息            ✅     614.9 KB     source_repos/LocalTransform/data/USPTO_480k/template_infos.csv
模型配置            ✅     136 B        source_repos/LocalTransform/data/configs/default_config.json
测试数据            ✅     18.9 MB      source_repos/LocalTransform/data/USPTO_480k/preprocessed_test.csv


In [10]:
# ====== Step 6.2: 查看模型配置 ======
import json

config_path = SOURCE_REPO / 'data' / 'configs' / 'default_config.json'
with open(config_path) as f:
    config = json.load(f)

print('LocalTransform 默认超参数:')
print(json.dumps(config, indent=2))

print(f'\n超参数说明:')
param_desc = {
    'attention_heads': '多头注意力的头数',
    'attention_layers': '全局反应性注意力的层数',
    'edge_hidden_feats': 'MPNN 边隐藏层维度',
    'node_out_feats': 'MPNN 节点输出维度（也是键网络和注意力的维度）',
    'num_step_message_passing': 'MPNN 消息传递轮数',
}
for k, v in config.items():
    desc = param_desc.get(k, '')
    print(f'  {k}: {v}  — {desc}')

LocalTransform 默认超参数:
{
  "attention_heads": 8,
  "attention_layers": 3,
  "edge_hidden_feats": 32,
  "node_out_feats": 256,
  "num_step_message_passing": 3
}

超参数说明:
  attention_heads: 8  — 多头注意力的头数
  attention_layers: 3  — 全局反应性注意力的层数
  edge_hidden_feats: 32  — MPNN 边隐藏层维度
  node_out_feats: 256  — MPNN 节点输出维度（也是键网络和注意力的维度）
  num_step_message_passing: 3  — MPNN 消息传递轮数


In [11]:
# ====== Step 6.3: 查看模板统计 ======
import pandas as pd

data_dir = SOURCE_REPO / 'data' / 'USPTO_480k'

real_df = pd.read_csv(data_dir / 'real_templates.csv')
virtual_df = pd.read_csv(data_dir / 'virtual_templates.csv')
info_df = pd.read_csv(data_dir / 'template_infos.csv')

print('模板统计:')
print(f'  真实键模板 (Real Templates): {len(real_df)} 种')
print(f'  虚拟键模板 (Virtual Templates): {len(virtual_df)} 种')
print(f'  全局模板信息: {len(info_df)} 条')

print(f'\n真实键模板示例 (前5个):')
print(real_df.head())

print(f'\n虚拟键模板示例 (前5个):')
print(virtual_df.head())

print(f'\n> 真实键模板: 描述键断裂(B)、键变化(C)、原子移除(R)等')
print(f'> 虚拟键模板: 描述键形成(A)操作')
print(f'> 总分类数: 真实键 {len(real_df)+1} 类(含类0=无反应), 虚拟键 {len(virtual_df)+1} 类')

模板统计:
  真实键模板 (Real Templates): 4370 种
  虚拟键模板 (Virtual Templates): 2535 种
  全局模板信息: 2846 条

真实键模板示例 (前5个):
   Unnamed: 0                                           Template  Frequency  \
0           0  [A:1].[A:2]#[A:3]>>[A:1]-[A:2]=[A:3]_-202_-101...          1   
1           1  [A:3]=[A:1]-[A:2](-[A:4])-[A:5]>>[A:1]#[A:2]_0...          1   
2           2  [A:3]=[A:1]-[A:2](-[A:4])-[A:5]>>[A:1]#[A:2]_0...          1   
3           3          [A:1]-[A:2]-[A:3]>>[A:1].[A:2]_12_00_00_B          1   
4           4  [A:4].[A:3].[A:5]-[A:1]=[A:2]>>[A:1]-[A:2]-[A:...          1   

   Class  
0      1  
1      2  
2      3  
3      4  
4      5  

虚拟键模板示例 (前5个):
   Unnamed: 0                                           Template  Frequency  \
0           0  [A:1].[A:2]#[A:3]>>[A:1]-[A:2]=[A:3]_-202_-101...          1   
1           1  [A:4].[A:3].[A:5]-[A:1]=[A:2]>>[A:1]-[A:2]-[A:...          1   
2           2  [A:1].[A:4].[A:2]-[A:3](=[A:5])-[A:6]>>[A:1]-[...          1   
3           3  [A:4

## 7. 快速冒烟测试：用预训练模型做一次预测

### 为什么在环境配置阶段就做预测？

这能一次性验证所有依赖（PyTorch、DGL、DGL-Life、RDKit）和数据文件都正确，同时让读者直观感受 LocalTransform 能做什么。

### 源码对应
- 文件: `Synthesis.py`
- 类: `localtransform`
- 方法: `predict_product()`

In [1]:
# ====== Step 7.1: 加载预训练模型并做预测 ======
import os, sys
from pathlib import Path

def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'LocalTransform'

# 切换到源码目录（因为 Synthesis.py 使用相对路径加载数据）
os.chdir(SOURCE_REPO)
sys.path.insert(0, str(SOURCE_REPO))
sys.path.insert(0, str(SOURCE_REPO / 'scripts'))

from Synthesis import localtransform

# 用 CPU 加载模型（教学用途不需要 GPU）
model = localtransform(dataset='USPTO_480k', device='cpu')
print('\n✅ 模型加载成功！')

/home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/localtransform_tutorial_envs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded 4370 real templates
loaded 2535 virtual templates


Parameters of loaded LocalTransform:
{'attention_heads': 8, 'attention_layers': 3, 'edge_hidden_feats': 32, 'node_out_feats': 256, 'num_step_message_passing': 3, 'Template_rn': 4370, 'Template_vn': 2535, 'in_node_feats': 80, 'in_edge_feats': 13}

✅ 模型加载成功！


In [2]:
# ====== Step 7.2: 预测一个简单反应 ======
# 示例: Suzuki 偶联反应 — 芳基硼酸 + 芳基溴化物
reactants = 'c1ccc(B(O)O)cc1.c1ccc(Br)cc1'
print(f'反应物: {reactants}')
print(f'含义: 苯基硼酸 + 溴苯\n')

results_df, results_dict = model.predict_product(reactants, topk=5, verbose=1)

print(f'\n预测结果:')
for k in range(1, 6):
    key = f'Top-{k}'
    if reactants in results_dict and key in results_dict[reactants]:
        pred = results_dict[reactants][key]
        product = pred.get('product', '')
        score = pred.get('score', 0)
        print(f'  {key}: {product}  (score: {score:.4f})')

反应物: c1ccc(B(O)O)cc1.c1ccc(Br)cc1
含义: 苯基硼酸 + 溴苯

1th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] B [np.int64(12), np.int64(13)] 0.98310596
2th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] A [np.int64(3), np.int64(12)] 0.9733487
3th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] B [np.int64(3), np.int64(4)] 0.96631926
4th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] A [np.int64(12), np.int64(3)] 0.9658849
5th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] A [np.int64(12), np.int64(4)] 0.016261946
6th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] B [np.int64(4), np.int64(3)] 0.008621778
7th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] A [np.int64(4), np.int64(12)] 0.0072661918
8th prediction: [A:1]-[A:3].[A:2]-[A:4]>>[A:1]-[A:2] B [np.int64(13), np.int64(12)] 0.0057319524
9th prediction: [A:1].[A:2]-[A:3]>>[A:1]-[A:2] A [np.int64(9), np.int64(12)] 0.0032210082
10th prediction: [A:1].[A:2]-[A:3]>>[A:1]-[A:2] A [np.int64(15), np.int64(12)] 0.002051736
11th predictio

In [3]:
# ====== Step 7.3: 再试几个反应 ======
test_reactions = [
    ('酰胺化', 'CC(=O)Cl.NCC'),
    ('醚化', 'CCO.CBr'),
    ('还原胺化', 'CC=O.NCC'),
]

for name, rxn in test_reactions:
    print(f'\n--- {name} ---')
    print(f'反应物: {rxn}')
    try:
        _, res = model.predict_product(rxn, topk=3, verbose=0)
        for k in range(1, 4):
            key = f'Top-{k}'
            if rxn in res and key in res[rxn]:
                pred = res[rxn][key]
                print(f'  {key}: {pred.get("product", "N/A")}  (score: {pred.get("score", 0):.4f})')
    except Exception as e:
        print(f'  预测失败: {e}')


--- 酰胺化 ---
反应物: CC(=O)Cl.NCC
  Top-1: CCNC(C)=O  (score: 0.9472)
  Top-2: CC(=O)CCN  (score: 0.4931)
  Top-3: CC(=O)C(C)N  (score: 0.4897)

--- 醚化 ---
反应物: CCO.CBr
  Top-1: CCOC  (score: 0.6665)
  Top-2: CC(C)O  (score: 0.3891)
  Top-3: CCCO  (score: 0.3736)

--- 还原胺化 ---
反应物: CC=O.NCC
  Top-1: CCNCC  (score: 0.7137)
  Top-2: CCCCN  (score: 0.3929)
  Top-3: CC=NCC  (score: 0.2189)


## 总结

### 本 notebook 完成了什么

| 步骤 | 内容 | 状态 |
|------|------|------|
| 1 | 了解 LocalTransform 的核心依赖 | ✅ |
| 2 | 创建并安装 conda 虚拟环境 | ✅ |
| 3 | 验证所有核心库（PyTorch CPU, DGL, DGL-Life, RDKit） | ✅ |
| 4 | 了解源码目录结构和模块职责 | ✅ |
| 5 | 理解核心概念（虚拟键/真实键、原子距离矩阵） | ✅ |
| 6 | 验证数据文件和模型权重 | ✅ |
| 7 | 用预训练模型完成冒烟测试 | ✅ |

### 关键概念回顾

- **真实键 (Real bonds)**: 分子中已存在的化学键，对应键断裂/变化/原子移除操作
- **虚拟键 (Virtual bonds)**: 未成键的原子对，对应键形成操作
- **原子距离矩阵 (ADM)**: 拓扑距离截断到 6，用作注意力的位置编码
- **局部模板 (Local Template)**: 描述单条键上发生的原子/键变化，而非整个反应的全局模板

### 源码与教学版差异

| 方面 | 原仓库 | 教学版 |
|------|--------|--------|
| GPU | CUDA 加速 | CPU 版本 |
| 训练 | 完整 Train.py | 不做训练 |
| 数据 | 完整 USPTO_480k | 使用预处理好的数据 |

### 下一步

→ **Notebook 2: 数据处理** — 了解从原始反应 SMILES 到模型输入的完整数据处理流程